## Product Listing Generator

In [3]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
import os
from dotenv import load_dotenv
from openai import OpenAI

In [6]:
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [ ]:
products_df.head"

#### Encoding Images for API

In [7]:
# Save each image as a .jpg file and add the missing columns
placeholder_prices = {
    "Apparel": 29.99,
    "Accessories": 24.99,
    "Footwear": 59.99,
    "Personal Care": 14.99,
}

image_paths = []
prices = []

for index, row in products_df.iterrows():
    image_path = images_dir / f"{row['id']}.jpg"
    row["image"].convert("RGB").save(image_path)
    image_paths.append(str(image_path))
    prices.append(placeholder_prices.get(row["masterCategory"], 19.99))

products_df["image_path"] = image_paths
products_df["price"] = prices
products_df["name"] = products_df["productDisplayName"]
products_df["category"] = products_df["masterCategory"] + " - " + products_df["articleType"]

print(f"✓ Saved {len(image_paths)} images to '{images_dir}'")
products_df[["id", "name", "price", "category", "image_path"]].head()

✓ Saved 100 images to 'product_images'


,id,name,price,category,image_path
0,15970,Turtle Check Men Navy Blue Shirt,29.99,Apparel - Shirts,product_images/15970.jpg
1,39386,Peter England Men Party Blue Jeans,29.99,Apparel - Jeans,product_images/39386.jpg
2,59263,Titan Women Silver Watch,24.99,Accessories - Watches,product_images/59263.jpg
3,21379,Manchester United Men Solid Black Track Pants,29.99,Apparel - Track Pants,product_images/21379.jpg
4,53759,Puma Men Grey T-shirt,29.99,Apparel - Tshirts,product_images/53759.jpg


In [8]:
import base64

def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    return encoded

# Example usage
sample_path = products_df.iloc[0]["image_path"]
encoded_image = encode_image_to_base64(sample_path)
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


### Creating the Product Lisitng Prompt

In [9]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 

print(test_prompt[:500] + "...")  # Show first 500 characters

You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


### Calling the ChatGPT API with Vision

In [ ]:
# Load OPENAI_API_KEY from the .env file next to this notebook
load_dotenv()

# Initialize client (reads OPENAI_API_KEY from the environment)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [37]:
img_data = encode_image_to_base64(sample_path)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "test_prompt"},
            {"type": "image_url",
             "image_url": {"url": f"data:image/jpeg;base64,{img_data}"}},
        ],
    }],
)

print(response.choices[0].message.content)

I can't identify the person in the image. However, I can help with general fashion advice or discuss styles similar to what you see!


In [11]:
def generate_listing(row):
    prompt = create_product_listing_prompt(row["name"], row["price"], row["category"])
    img_data = encode_image_to_base64(row["image_path"])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=1000,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_data}"}},
        ]}],
    )
    return response.choices[0].message.content

products_df["listing"] = products_df.apply(generate_listing, axis=1)

In [14]:
products_df["listing"]

0     ```json\n{\n    "title": "Stylish Turtle Check...
1     ```json\n{\n    "title": "Stylish Peter Englan...
2     ```json\n{\n    "title": "Elegant Titan Women ...
3     ```json\n{\n    "title": "Manchester United Me...
4     ```json\n{\n    "title": "Stylish Puma Men Gre...
                            ...                        
95    ```json\n{\n    "title": "Chic Green Printed N...
96    ```json\n{\n    "title": "Stylish Indigo Natio...
97    ```json\n{\n    "title": "DKNY Women Red Delic...
98    ```json\n{\n    "title": "Urban Yoga Women's B...
99    ```json\n{\n    "title": "Stylish Gini and Jon...
Name: listing, Length: 100, dtype: str